# 02 · Clean, audit and engineer
**Rules (PROTOCOL.md §5):** decode fills only · flag, never delete extremes · causal imputation only · target never imputed.
**EDA discipline:** Dst-based plots use data up to 2020 only. The test period's labels are not inspected.

In [ ]:
import os, sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "stormtime").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "stormtime").exists(), "Run inside the cloned repo (see README)"
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stormtime import config as C, data as D, features as F, events as E, splits as S, baselines as B, metrics as M
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
RAW, CLEAN, FIG, RES = ROOT/"data/raw", ROOT/"data/clean", ROOT/"figures", ROOT/"results"
for p in (RAW, CLEAN, FIG, RES): p.mkdir(parents=True, exist_ok=True)
print("repo:", ROOT)

In [ ]:
man = json.load(open(RAW / "manifest.json"))
avail = {k: tuple(v) for k, v in man["documentation"]["availability"].items()}
YEARS = [r["year"] for r in man["files"]]
raw = D.add_time(D.load_raw(RAW, YEARS))
last_obs = raw[["speed", "bz_gsm", "dst"]].dropna(how="all").index.max()   # OMNI pads the current year with fill rows
print("last hour with any observation:", last_obs, "| trimming", int((raw.index > last_obs).sum()), "trailing fill rows")
raw = raw.loc[:last_obs]
lv = D.label_version(raw.index, avail)
base = F.select_clean(raw, lv)
print(base.shape, "| label tiers:", base["label_version"].value_counts().to_dict())

## 1 · Missingness (inputs only; no label values used)

In [ ]:
cols = C.INPUT_COLS + ["dst"]
yr = base.index.year
miss = pd.DataFrame({c: base[c].isna().groupby(yr).mean() for c in cols}).T
fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(miss.values, aspect="auto", cmap="magma_r", vmin=0, vmax=0.5)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols); ax.set_xticks(range(0, len(miss.columns), 3))
ax.set_xticklabels(miss.columns[::3]); ax.grid(False); plt.colorbar(im, label="fraction missing")
ax.set_title("Missingness by variable and year"); plt.tight_layout(); plt.savefig(FIG / "02_missingness_heatmap.png"); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3))
for ax, c in zip(axs, ["dst", "speed", "bz_gsm"]):
    g = F.gap_lengths(base[c])
    if len(g): ax.hist(g, bins=np.logspace(0, np.log10(max(g.max(), 10)), 30)); ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(f"{c}: gap lengths (h)\n{len(g)} gaps, longest {g.max() if len(g) else 0} h")
plt.tight_layout(); plt.savefig(FIG / "02_gap_lengths.png"); plt.show()

## 2 · Distributions and soft-bound flags (flag only: valid extremes are kept)

In [ ]:
dev = base.loc[:"2020-12-31"]
fig, axs = plt.subplots(1, 4, figsize=(13, 3))
for ax, c in zip(axs, ["speed", "density", "bz_gsm", "dst"]):
    ax.hist(dev[c].dropna(), bins=80); ax.set_title(c); ax.set_yscale("log")
for thr in C.CLASS_EDGES: axs[3].axvline(thr, color="r", lw=.8)
plt.tight_layout(); plt.savefig(FIG / "02_distributions.png"); plt.show()
flags = {c: int(base[c].sum()) for c in base.columns if c.startswith("flag_")}
print("flagged (NOT removed):", flags)

## 3 · Consistency checks against OMNI's own derived fields

In [ ]:
f0 = F.add_physics(base)
ok = f0["vbs"].notna() & f0["e_field"].notna()
d = (f0.loc[ok, "vbs"] - np.maximum(f0.loc[ok, "e_field"], 0)).abs()
print(f"VBs vs max(E_field,0): median |diff| {d.median():.4f}, 99.9% {d.quantile(.999):.3f} mV/m (n={ok.sum()})")
p_re = 2.0e-6 * base["density"] * base["speed"] ** 2
okp = p_re.notna() & base["pressure"].notna()
rel = ((base.loc[okp, "pressure"] - p_re[okp]) / base.loc[okp, "pressure"].clip(lower=.1)).abs()
print(f"OMNI pressure vs 2e-6*N*V^2: median rel diff {rel.median():.3f} (alpha particles explain a few % excess)")
dev_dst = base.loc[:"2020-12-31"]
print("Spearman(Kp, -Dst) on <=2020:", round(dev_dst[["kp10", "dst"]].dropna().corr(method="spearman").iloc[0, 1] * -1, 3))

## 4 · Imputation policies: what does each rule remove, and does it hit storms harder than quiet times?

In [ ]:
core = ["speed", "bz_gsm", "pressure"]
dst_dev = base["dst"]
storm = (dst_dev <= -50) & (base.index <= "2020-12-31")
quiet = (dst_dev > -30) & (base.index <= "2020-12-31")
rows = []
for name, k in [("mask only", 0), ("ffill<=3h", 3), ("ffill<=6h", 6)]:
    imp = F.causal_impute(base, core, k)
    complete = imp[core].notna().all(axis=1)
    rows.append({"policy": name, "complete_all_hours": complete.mean(), "complete_storm_hours": complete[storm].mean(),
                 "complete_quiet_hours": complete[quiet].mean(), "n_storm_hours": int(storm.sum())})
pol = pd.DataFrame(rows).set_index("policy").round(4); pol

Default policy: **ffill ≤ 3 h** (a hypothesis, not an optimum; compared again as an ablation later). Storm vs quiet completeness above shows whether cleaning removes storm data disproportionately.

In [ ]:
pol.to_csv(RES / "imputation_policies.csv")
imp = F.causal_impute(base, C.INPUT_COLS, 3)
clean = F.add_physics(imp)
n_valid_dstar = int(clean.loc[:"2020-12-31", "dst_star"].notna().sum())
print("clean:", clean.shape, "| hours with valid Dst* (<=2020):", n_valid_dstar)

In [ ]:
def save_parquet(df, path):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == "float64": out[c] = out[c].astype("float32")
    out["label_version"] = out["label_version"].astype(str)
    out.to_parquet(path, compression="zstd")
save_parquet(base, CLEAN / "omni_hourly_base.parquet")
save_parquet(clean, CLEAN / "omni_hourly_clean.parquet")
report = {"rows": int(len(clean)), "start": str(clean.index[0]), "end": str(clean.index[-1]),
          "label_tiers": clean["label_version"].astype(str).value_counts().to_dict(),
          "flags_not_removed": flags, "imputation_default": "ffill<=3h, causal, masks + age columns",
          "policies": pol.to_dict(), "columns": list(clean.columns)}
json.dump(report, open(RES / "cleaning_report.json", "w"), indent=1, default=str)
print({p.name: f"{p.stat().st_size/1e6:.1f} MB" for p in CLEAN.glob("*.parquet")})

## 5 · Storm context (pre-2021 only)

In [ ]:
dev = clean.loc[:"2020-12-31"]
ev = E.find_events(dev["dst"].to_numpy(), -100)
top = ev.sort_values("min_dst").head(4)
fig, axs = plt.subplots(len(top), 1, figsize=(10, 2.3 * len(top)), sharex=False)
axs = np.atleast_1d(axs)
for ax, (_, r) in zip(axs, top.iterrows()):
    s, e = int(r.start_idx) - 48, int(r.end_idx) + 72
    seg = dev.iloc[max(s, 0):e]
    ax.plot(seg.index, seg["dst"], color="k", label="Dst"); ax.set_ylabel("Dst (nT)")
    a2 = ax.twinx(); a2.plot(seg.index, seg["vbs"], color="tab:orange", alpha=.7); a2.set_ylabel("VBs (mV/m)", color="tab:orange"); a2.grid(False)
    ax.set_title(f"min Dst {r.min_dst:.0f} nT at {dev.index[int(r.min_idx)]:%Y-%m-%d %H:%M}", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "02_top_storms_pre2021.png"); plt.show()

## 6 · Spot-check against the official source
Pre-2021 storms only. Compare these minima with WDC Kyoto's own Dst listing (https://wdc.kugi.kyoto-u.ac.jp/dst_final/) for the same hours; any mismatch means a parsing or timing error.

In [ ]:
windows = {"Halloween 2003": ("2003-10-28", "2003-11-02"), "Nov 2003": ("2003-11-19", "2003-11-23"),
           "St Patrick's Day 2015": ("2015-03-16", "2015-03-19"), "Sep 2017": ("2017-09-06", "2017-09-10")}
rows = []
for k, (a, b_) in windows.items():
    seg = clean.loc[a:b_, "dst"]
    rows.append({"storm": k, "min_Dst_nT": seg.min(), "at_UTC_hour_start": seg.idxmin(), "label_version": clean.loc[seg.idxmin(), "label_version"]})
pd.DataFrame(rows).set_index("storm")

### Commit
```
git add data/clean results figures && git commit -m "Clean OMNI table + EDA" && git push
```